# Phase 7 — Effect of image enhancement on object detection in fog and low light

**Research question 1.** Does placing an image-enhancement model in front of a COCO-pretrained detector improve detection accuracy on real degraded images?

**Research question 3.** How do detectors trade accuracy for speed under the same conditions, and does the effect of enhancement hold across detectors?

**Hypothesis.** A task-specific enhancer (FFA-Net for haze, Zero-DCE++ for low light) raises COCO mAP compared with no enhancement and with a classical baseline (CLAHE).

## Experimental design

| | |
|---|---|
| Framework | MMDetection 3.3 (`Runner.test()` + `CocoMetric`), mmcv 2.1, torch 2.1, CUDA |
| Main detector | RTMDet-s (`rtmdet_s_8xb32-300e_coco`), official OpenMMLab weights, no fine-tuning |
| Comparison detectors | RTMDet-tiny, Faster R-CNN R50-FPN, RetinaNet R50-FPN, SSD300 |
| Fog data | RTTS (real hazy scenes), test split, 433 images, 4,500 boxes, 5 classes |
| Low-light data | ExDark (real night scenes), test split, 734 images, 2,424 boxes, 12 classes |
| Enhancement, fog | none · CLAHE · FFA-Net (outdoor weights) |
| Enhancement, low light | none · CLAHE · Zero-DCE++ |
| Accuracy metrics | COCO mAP@[.5:.95], mAP@0.5, mAP@0.75, mAP by object size, per-class AP |
| Cost metrics | enhancement ms/img, detector ms/img, FPS, parameters, peak GPU memory |

**Controlled variables.** The detector weights, the images, the image resolution and the evaluation code are identical in every run. Only the enhancement step changes, so a difference in mAP comes from the enhancement.

**Class mapping.** Dataset labels are mapped to their COCO names (for example *motorbike* → *motorcycle*, *People* → *person*), so the COCO-pretrained detectors are scored without retraining. A prediction with the wrong COCO class (for example *truck* for a box labelled *car*) counts as a miss. This applies equally to every enhancement method.

**Data notes.** The ExDark copy is a Roboflow export with images resized to 640×640 and the original labels. Speed is measured on the Colab GPU (Tesla T4).

## 1. Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/computer_vision_expirement'
RESULTS = '/content/drive/MyDrive/computer_vision/results/mmdet_eval'
OUT = f'{RESULTS}/mmdet_eval.jsonl'
os.makedirs(RESULTS, exist_ok=True)

if not os.path.isdir(REPO):
    !git clone -q https://github.com/Ib-Programmer/computer_vision_expirement.git {REPO}
%cd {REPO}
!git checkout -q main && git pull -q origin main && git log --oneline -1

MMDetection 3.3 needs mmcv 2.1, whose CUDA wheels exist only for Python ≤ 3.11 and torch 2.1. Colab runs Python 3.12, so the detector runs in a separate Python 3.10 environment built with `uv`. `MPLBACKEND=Agg` is set for that environment because Colab's inline plotting backend does not exist there.

In [ ]:
ENV = '/content/mmenv'
PY = f'{ENV}/bin/python'
MM = f'MPLBACKEND=Agg {PY}'   # interpreter of the mmdet environment
MMCV_WHL = ('https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/'
            'mmcv-2.1.0-cp310-cp310-manylinux1_x86_64.whl')
if not os.path.exists(PY):
    !pip install -q uv
    !uv venv {ENV} --python 3.10 -q
    !uv pip install -q --python {PY} torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
    !uv pip install -q --python {PY} "numpy<2" "opencv-python<4.11" mmengine==0.10.7 mmdet==3.3.0 {MMCV_WHL} pycocotools pyyaml einops huggingface_hub gdown
!{MM} -c "import torch, mmcv, mmdet, mmcv.ops; print('torch', torch.__version__, '| mmcv', mmcv.__version__, '| mmdet', mmdet.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"

## 2. Enhancement weights and evaluation data

Zero-DCE++ and FFA-Net weights come from the project's Hugging Face model repo. RTTS (`fuhao7i/RTTS`) and ExDark (`dronefreak/ExDark`) are converted to COCO-format annotation files with COCO class names.

In [ ]:
from huggingface_hub import hf_hub_download
os.makedirs(f'{REPO}/weights', exist_ok=True)
for f in ['zero_dce_pp.pth', 'ffa_net_outdoor.pk']:
    hf_hub_download('IbProgrammmer/cv-thesis-models', f, local_dir=f'{REPO}/weights')

!python scripts/prepare_eval_datasets.py --dataset all --split test

## 3. Experiment 1 — RTMDet-s with each enhancement method

Six runs: three enhancement settings on each dataset. Each run appends one result line to `mmdet_eval.jsonl` on Drive. A run that is already in the file is skipped, so the cell continues where it stopped after a disconnect.

In [ ]:
MODEL = 'rtmdet_s_8xb32-300e_coco'
MATRIX = {
    'rtts':   ['none', 'clahe', 'ffanet'],
    'exdark': ['none', 'clahe', 'zero_dce'],
}

for dataset, methods in MATRIX.items():
    for enh in methods:
        print(f'\n===== {MODEL} | {dataset} | {enh} =====')
        !{MM} scripts/mmdet_eval.py --model {MODEL} --dataset {dataset} --enhance {enh} --out {OUT} --skip-existing

## 4. Experiment 2 — Detector comparison

Four more COCO-pretrained MMDetection detectors are evaluated on the same images, without enhancement and with each dataset's task-specific enhancer. This shows whether the effect from Experiment 1 holds for other architectures, and gives the accuracy-versus-speed trade-off.

In [ ]:
DETECTORS = [
    'rtmdet_tiny_8xb32-300e_coco',
    'faster-rcnn_r50_fpn_1x_coco',
    'retinanet_r50_fpn_1x_coco',
    'ssd300_coco',
]
MAIN_ENHANCER = {'rtts': 'ffanet', 'exdark': 'zero_dce'}

for m in DETECTORS:
    for dataset, enh in MAIN_ENHANCER.items():
        for e in ['none', enh]:
            print(f'\n===== {m} | {dataset} | {e} =====')
            !{MM} scripts/mmdet_eval.py --model {m} --dataset {dataset} --enhance {e} --out {OUT} --skip-existing

## 5. Results

**Δ mAP** is the change in percentage points against no enhancement, for the same detector and dataset. All tables are also saved to Drive as CSV and Markdown.

In [ ]:
!pip install -q tabulate
import json
import pandas as pd

assert os.path.exists(OUT), f'{OUT} not found - the evaluation runs above did not finish; check their output.'
rows = [json.loads(l) for l in open(OUT) if l.strip()]
df = pd.DataFrame(rows)
df = df.sort_values('time').drop_duplicates(['model', 'dataset', 'enhance'], keep='last')
df = df.sort_values(['model', 'dataset', 'time'], kind='stable').reset_index(drop=True)

base = df[df.enhance == 'none'].set_index(['model', 'dataset'])
for m in ['mAP', 'mAP50', 'mAP75']:
    df[f'd_{m}'] = df.apply(lambda r: r[m] - base[m].get((r.model, r.dataset), float('nan')), axis=1)

pct = lambda x: None if pd.isna(x) else round(100 * x, 1)
acc = df[['model', 'dataset', 'enhance', 'n_images', 'mAP', 'mAP50', 'mAP75', 'mAP_s', 'mAP_m', 'mAP_l',
          'd_mAP', 'd_mAP50', 'enhance_ms_per_img']].copy()
for c in ['mAP', 'mAP50', 'mAP75', 'mAP_s', 'mAP_m', 'mAP_l', 'd_mAP', 'd_mAP50']:
    acc[c] = acc[c].map(pct)
acc = acc.rename(columns={'mAP': 'mAP %', 'mAP50': 'mAP@0.5 %', 'mAP75': 'mAP@0.75 %',
                          'mAP_s': 'mAP small %', 'mAP_m': 'mAP medium %', 'mAP_l': 'mAP large %',
                          'd_mAP': 'Δ mAP (pts)', 'd_mAP50': 'Δ mAP@0.5 (pts)',
                          'enhance_ms_per_img': 'enhance ms/img'})

table1 = acc[acc.model == MODEL].drop(columns='model')
print(f'Table 1 - {MODEL}: detection accuracy with and without enhancement')
display(table1)

table2 = acc[acc.model != MODEL]
print('\nTable 2 - Detector comparison: no enhancement vs. task-specific enhancer')
display(table2)

speed_cols = [c for c in ['model', 'dataset', 'det_ms_per_img', 'fps', 'params_M', 'gpu_mem_MB', 'zoo_coco_box_AP', 'device']
              if c in df.columns]
table3 = df[df.enhance == 'none'][speed_cols].merge(
    acc[acc.enhance == 'none'][['model', 'dataset', 'mAP %', 'mAP@0.5 %']], on=['model', 'dataset'])
print('\nTable 3 - Detector cost and accuracy (no enhancement)')
display(table3)

table4 = df.set_index(['model', 'dataset', 'enhance'])['per_class_AP'].apply(pd.Series).mul(100).round(1)
print('\nTable 4 - Per-class AP %')
display(table4)

table1.to_csv(f'{RESULTS}/table1_rtmdet_s_enhancement.csv', index=False)
table2.to_csv(f'{RESULTS}/table2_detector_comparison.csv', index=False)
table3.to_csv(f'{RESULTS}/table3_detector_cost.csv', index=False)
table4.to_csv(f'{RESULTS}/table4_per_class_ap.csv')
with open(f'{RESULTS}/tables.md', 'w', encoding='utf-8') as f:
    f.write(f'## Table 1 - {MODEL}: detection accuracy with and without enhancement\n\n' + table1.to_markdown(index=False) + '\n\n')
    f.write('## Table 2 - Detector comparison: no enhancement vs. task-specific enhancer\n\n' + table2.to_markdown(index=False) + '\n\n')
    f.write('## Table 3 - Detector cost and accuracy (no enhancement)\n\n' + table3.to_markdown(index=False) + '\n\n')
    f.write('## Table 4 - Per-class AP %\n\n' + table4.to_markdown() + '\n')
print('\nSaved to', RESULTS)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

TITLES = {'rtts': 'RTTS (real fog)', 'exdark': 'ExDark (low light)'}
SHORT = {'rtmdet_s_8xb32-300e_coco': 'RTMDet-s', 'rtmdet_tiny_8xb32-300e_coco': 'RTMDet-tiny',
         'faster-rcnn_r50_fpn_1x_coco': 'Faster R-CNN', 'retinanet_r50_fpn_1x_coco': 'RetinaNet',
         'ssd300_coco': 'SSD300'}

# Figure 1: RTMDet-s mAP and mAP@0.5 per enhancement method
main = df[df.model == MODEL]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, dataset in zip(axes, MATRIX):
    d = main[main.dataset == dataset].set_index('enhance').reindex(MATRIX[dataset]).dropna(subset=['mAP'])
    x = range(len(d))
    ax.bar([i - 0.2 for i in x], d.mAP * 100, 0.4, label='mAP@[.5:.95]', color='#4C72B0')
    ax.bar([i + 0.2 for i in x], d.mAP50 * 100, 0.4, label='mAP@0.5', color='#DD8452')
    for i, (a, b) in enumerate(zip(d.mAP * 100, d.mAP50 * 100)):
        ax.text(i - 0.2, a + 0.5, f'{a:.1f}', ha='center', fontsize=8)
        ax.text(i + 0.2, b + 0.5, f'{b:.1f}', ha='center', fontsize=8)
    ax.set_xticks(list(x), d.index)
    ax.set_title(TITLES[dataset])
    ax.set_xlabel('enhancement')
axes[0].set_ylabel('AP (%)')
axes[0].legend()
fig.suptitle(f'{MODEL}: effect of enhancement')
fig.tight_layout()
fig.savefig(f'{RESULTS}/fig1_enhancement_map.png', dpi=200)
plt.show()

# Figure 2: change in mAP from the task-specific enhancer, per detector
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (dataset, enh) in zip(axes, MAIN_ENHANCER.items()):
    d = df[(df.dataset == dataset) & (df.enhance == enh)].set_index('model')['d_mAP'].dropna() * 100
    ax.barh([SHORT.get(m, m) for m in d.index], d.values, color=['#55A868' if v >= 0 else '#C44E52' for v in d.values])
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{TITLES[dataset]}: {enh} vs. none')
    ax.set_xlabel('Δ mAP (points)')
fig.tight_layout()
fig.savefig(f'{RESULTS}/fig2_delta_map_per_detector.png', dpi=200)
plt.show()

# Figure 3: accuracy vs. detector speed (no enhancement)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, dataset in zip(axes, MATRIX):
    d = df[(df.dataset == dataset) & (df.enhance == 'none')]
    ax.scatter(d.fps, d.mAP * 100, s=d.params_M * 4 + 20, alpha=0.7)
    for _, r in d.iterrows():
        ax.annotate(SHORT.get(r.model, r.model),(r.fps, r.mAP * 100), fontsize=8, xytext=(4, 4), textcoords='offset points')
    ax.set_title(TITLES[dataset])
    ax.set_xlabel('detector FPS (T4)')
    ax.set_ylabel('mAP (%)')
fig.suptitle('Accuracy vs. speed (marker size = parameters)')
fig.tight_layout()
fig.savefig(f'{RESULTS}/fig3_accuracy_vs_speed.png', dpi=200)
plt.show()

## 6. Findings

The findings below are computed from the result file, so they always match the tables above.

In [ ]:
lines = []
for dataset, methods in MATRIX.items():
    d = df[(df.model == MODEL) & (df.dataset == dataset)].set_index('enhance')
    if 'none' not in d.index:
        continue
    b = d.loc['none']
    lines.append(f"**{TITLES[dataset]}**, {MODEL}: baseline mAP {b.mAP*100:.1f}, mAP@0.5 {b.mAP50*100:.1f}.")
    for enh in methods[1:]:
        if enh not in d.index:
            continue
        r = d.loc[enh]
        verdict = 'improves' if r.d_mAP > 0 else 'does not improve'
        lines.append(f"- {enh} {verdict} detection: Δ mAP {r.d_mAP*100:+.1f} pts, Δ mAP@0.5 {r.d_mAP50*100:+.1f} pts, "
                     f"at {r.enhance_ms_per_img:.1f} ms/img of enhancement.")
    pcd = table4.loc[(MODEL, dataset)]
    if 'none' in pcd.index and len(pcd) > 1:
        best = d.drop(index='none')['mAP'].idxmax()
        diff = (pcd.loc[best] - pcd.loc['none']).dropna().sort_values()
        lines.append(f"- Per class with {best}: largest gain {diff.index[-1]} ({diff.iloc[-1]:+.1f}), "
                     f"largest loss {diff.index[0]} ({diff.iloc[0]:+.1f}).")
    lines.append('')

for dataset, enh in MAIN_ENHANCER.items():
    d = df[(df.dataset == dataset) & (df.enhance == enh)].dropna(subset=['d_mAP'])
    if len(d):
        n_up = int((d.d_mAP > 0).sum())
        lines.append(f"**{TITLES[dataset]}**, {enh} across detectors: mAP rises for {n_up} of {len(d)} detectors "
                     f"(mean Δ {d.d_mAP.mean()*100:+.1f} pts, range {d.d_mAP.min()*100:+.1f} to {d.d_mAP.max()*100:+.1f}).")
        lines.append("")

t = df[df.enhance == 'none'].dropna(subset=['fps'])
if len(t):
    lines.append('')
    for dataset in MATRIX:
        d = t[t.dataset == dataset]
        if len(d):
            acc_best, fast = d.loc[d.mAP.idxmax()], d.loc[d.fps.idxmax()]
            lines.append(f"**{TITLES[dataset]}**: most accurate detector {acc_best.model} ({acc_best.mAP*100:.1f} mAP, {acc_best.fps:.0f} FPS); "
                         f"fastest {fast.model} ({fast.fps:.0f} FPS, {fast.mAP*100:.1f} mAP).")
            lines.append("")

from IPython.display import Markdown
findings = '\n'.join(lines)
with open(f'{RESULTS}/findings.md', 'w', encoding='utf-8') as f:
    f.write(findings + '\n')
display(Markdown(findings))